# 🎓 EduPath AI — GRPO Training
**Team KRIYA | Meta Hackathon 2026** | Runtime: T4 GPU

Environment-as-verifier: LLM generates tutoring actions → EduPath env scores them → GRPO updates policy.

In [ ]:
!pip install -q trl>=0.15.0 transformers accelerate peft datasets bitsandbytes scipy
!pip install -q --no-deps unsloth 2>/dev/null || echo 'Unsloth unavailable (OK)'

In [ ]:
import subprocess,os,sys
R='/content/edupath'
if not os.path.exists(R):
    subprocess.run(['git','clone','https://huggingface.co/spaces/degree-checker-01/meta-new-space',R],check=True)
os.chdir(R); sys.path.insert(0,f'{R}/backend')
from unittest.mock import MagicMock
for m in ['llm_blender','llm_blender.agents','llm_blender.pair_ranker']:
    if m not in sys.modules: sys.modules[m]=MagicMock()
from environment.env import EduPathEnv
from environment.models import Action, ActionType
from environment.student import student_manager
from environment.curriculum import TOPIC_GRAPH
env=EduPathEnv(seed=42)
s=student_manager.create(name='Demo')
student_manager.update_from_onboarding(s.id,{'target_field':'tech','learning_goal':'Python','weekly_hours':10})
obs=env.reset(student_id=s.id,seed=42)
print(f'Topics:{len(TOPIC_GRAPH)} Available:{obs.available_topics[:5]} \u2705')

## Reward Function + Dataset

In [ ]:
import json,re,random,logging
import numpy as np
logger=logging.getLogger('edupath')
_parse_fails=0

def edupath_reward(completions, **kwargs):
    global _parse_fails
    rewards=[]
    for comp in completions:
        if isinstance(comp,list):
            text=''
            for msg in comp:
                if isinstance(msg,dict) and msg.get('role')=='assistant': text=msg.get('content',''); break
            if not text and comp:
                last=comp[-1]; text=last.get('content','') if isinstance(last,dict) else str(last)
        elif isinstance(comp,dict): text=comp.get('content',str(comp))
        else: text=str(comp)
        try:
            match=re.search(r'\{[^}]+\}',text)
            if not match:
                _parse_fails+=1; rewards.append(-0.3); continue
            d=json.loads(match.group())
            atype=d.get('type','recommend_resource')
            # Format reward: +0.1 for valid JSON with 'type' key
            fmt_bonus=0.1 if 'type' in d else 0.0
            action=Action(type=ActionType(atype),topic_id=d.get('topic_id'))
            r=env._calculate_reward(action).value + fmt_bonus
            rewards.append(r)
        except Exception as e:
            _parse_fails+=1
            logger.warning(f'Reward parse error: {e} | text={text[:80]}')
            rewards.append(-0.3)
    return rewards

# Quick sanity check
tests=['{"type":"recommend_topic","topic_id":"python_basics"}','{"type":"mark_job_ready"}','garbage']
for a,r in zip(tests,edupath_reward(tests)): print(f'  {a[:50]:50s} r={r:+.2f}')
print('Reward function \u2705')

In [ ]:
# Build DIVERSE dataset (different profiles, fields, skill levels)
from datasets import Dataset
PROFILES=[
  {'field':'tech','goal':'ML Engineer','hours':10,'skills':[]},
  {'field':'tech','goal':'Data Analyst','hours':15,'skills':[{'skill':'Python','proficiency':0.4}]},
  {'field':'tech','goal':'Web Developer','hours':8,'skills':[{'skill':'JavaScript','proficiency':0.3}]},
  {'field':'healthcare','goal':'AI in Medicine','hours':10,'skills':[{'skill':'Biology','proficiency':0.8}]},
  {'field':'business','goal':'Business Analytics','hours':12,'skills':[{'skill':'Excel','proficiency':0.5}]},
]
def make_prompt(o):
    return f"""You are an AI tutoring agent. Analyze the student state and choose the best action.
STATE:
- Completed: {o.get('completed_topics',[])}
- Available: {o.get('available_topics',[])[:6]}
- Quiz scores: {o.get('quiz_history_summary',{})}
- Job readiness: {o.get('job_readiness_score',0):.2f}
- Current topic: {o.get('current_topic','None')}
ACTIONS: recommend_topic, assign_quiz, assign_mini_project, assign_capstone, recommend_resource, mark_job_ready
Respond ONLY with JSON: {{"type":"<action>","topic_id":"<id_or_null>"}}"""

train_prompts,eval_prompts=[],[]
for seed in range(500):
    p=PROFILES[seed%len(PROFILES)]
    e=EduPathEnv(seed=seed)
    s=student_manager.create(name=f's{seed}')
    onboard={'target_field':p['field'],'learning_goal':p['goal'],'weekly_hours':p['hours']}
    if p['skills']: onboard['skills']=p['skills']
    student_manager.update_from_onboarding(s.id,onboard)
    o=e.reset(student_id=s.id,seed=seed)
    prompt=make_prompt(o.model_dump())
    if seed<400: train_prompts.append(prompt)
    else: eval_prompts.append(prompt)  # held-out eval set

train_dataset=Dataset.from_dict({'prompt':train_prompts})
print(f'Train: {len(train_prompts)} | Eval (held-out): {len(eval_prompts)} \u2705')

## Load Model (Qwen2.5-1.5B-Instruct)

In [ ]:
import torch,inspect
from trl import GRPOTrainer,GRPOConfig
vram=torch.cuda.get_device_properties(0).total_memory/1e9
print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {vram:.1f}GB')

MODEL_ID='Qwen/Qwen2.5-1.5B-Instruct'
model=tokenizer=None
try:
    from unsloth import FastLanguageModel
    model,tokenizer=FastLanguageModel.from_pretrained('unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit',max_seq_length=1024,load_in_4bit=True)
    model=FastLanguageModel.get_peft_model(model,r=16,target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],lora_alpha=32,lora_dropout=0.05,bias='none',use_gradient_checkpointing='unsloth')
    print('Unsloth 4-bit \u2705')
except Exception as e:
    print(f'Unsloth failed: {e}')
    from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
    bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_use_double_quant=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.float16)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,quantization_config=bnb,device_map='auto')
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
    try:
        from peft import get_peft_model,LoraConfig,TaskType
        lora=LoraConfig(r=16,lora_alpha=32,target_modules=['q_proj','k_proj','v_proj','o_proj'],lora_dropout=0.05,bias='none',task_type=TaskType.CAUSAL_LM)
        model=get_peft_model(model,lora)
    except: pass
    print('HuggingFace 4-bit \u2705')
total=sum(p.numel() for p in model.parameters())
train_p=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Params: {total/1e6:.0f}M total, {train_p/1e6:.1f}M trainable')

## Baseline Evaluation (BEFORE training)

In [ ]:
def evaluate_model(model,tokenizer,prompts,label='Model',n=50):
    model.eval(); rewards_all=[]; actions_count={}
    valid_json=0
    for p in prompts[:n]:
        inp=tokenizer(p,return_tensors='pt',truncation=True,max_length=512).to(model.device)
        with torch.no_grad():
            out=model.generate(**inp,max_new_tokens=64,temperature=0.7,do_sample=True,pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
        text=tokenizer.decode(out[0][inp['input_ids'].shape[1]:],skip_special_tokens=True).strip()
        r=edupath_reward([text])[0]; rewards_all.append(r)
        m=re.search(r'\{[^}]+\}',text)
        if m:
            valid_json+=1
            try:
                d=json.loads(m.group()); at=d.get('type','unknown')
                actions_count[at]=actions_count.get(at,0)+1
            except: actions_count['parse_fail']=actions_count.get('parse_fail',0)+1
        else: actions_count['no_json']=actions_count.get('no_json',0)+1
    model.train()
    return {'label':label,'mean':float(np.mean(rewards_all)),'std':float(np.std(rewards_all)),
        'pos_rate':float(np.mean([r>0 for r in rewards_all])),'json_rate':valid_json/n,
        'actions':actions_count,'rewards':rewards_all}

print('Baseline eval (50 held-out samples)...')
baseline=evaluate_model(model,tokenizer,eval_prompts,'Baseline',50)
print(f"  Mean reward: {baseline['mean']:+.3f} \u00b1 {baseline['std']:.3f}")
print(f"  Positive %:  {baseline['pos_rate']:.0%}")
print(f"  Valid JSON:  {baseline['json_rate']:.0%}")
print(f"  Actions:     {baseline['actions']}")
print('Baseline recorded \u2705')

## GRPO Training (500 steps, checkpoints every 100)

In [ ]:
_sig=set(inspect.signature(GRPOConfig.__init__).parameters.keys())
args=dict(output_dir='edupath_grpo',max_steps=500,per_device_train_batch_size=2,
    gradient_accumulation_steps=4,learning_rate=5e-5,lr_scheduler_type='cosine',
    warmup_ratio=0.05,weight_decay=0.01,max_grad_norm=0.3,
    logging_steps=10,save_steps=100,save_total_limit=3,
    report_to='none',seed=42,
    bf16=torch.cuda.is_bf16_supported(),fp16=not torch.cuda.is_bf16_supported())
for k,v in {'num_generations':4,'max_prompt_length':512,'beta':0.04}.items():
    if k in _sig: args[k]=v
for c in ['max_completion_length','max_new_tokens','generation_max_new_tokens']:
    if c in _sig: args[c]=80; break

config=GRPOConfig(**args)
trainer=GRPOTrainer(model=model,args=config,train_dataset=train_dataset,
    processing_class=tokenizer,reward_funcs=[edupath_reward])

from datetime import datetime
print(f'Training {config.max_steps} steps (checkpoints every 100)...')
t0=datetime.now()
result=trainer.train()
mins=(datetime.now()-t0).total_seconds()/60
print(f'\nDone! {mins:.1f}min | loss={result.training_loss:.4f} | parse_fails={_parse_fails} \u2705')

## Post-Training Evaluation + Comparison

In [ ]:
print('Post-training eval (50 held-out samples)...')
trained=evaluate_model(model,tokenizer,eval_prompts,'Trained',50)
print(f"  Mean reward: {trained['mean']:+.3f} \u00b1 {trained['std']:.3f}")
print(f"  Positive %:  {trained['pos_rate']:.0%}")
print(f"  Valid JSON:  {trained['json_rate']:.0%}")
print(f"  Actions:     {trained['actions']}")
imp=trained['mean']-baseline['mean']
print(f'\n  Improvement: {imp:+.3f} ({"\u2191 BETTER" if imp>0 else "\u2193 WORSE"})')

## Training Curves + Before/After + Action Distribution

In [ ]:
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d
log=trainer.state.log_history
st=[e['step'] for e in log if 'loss' in e]
lo=[e['loss'] for e in log if 'loss' in e]
rs=[e['step'] for e in log if 'reward' in e]
rw=[e['reward'] for e in log if 'reward' in e]

fig,axes=plt.subplots(2,2,figsize=(16,10))

# 1. Loss
axes[0,0].plot(st,lo,'royalblue',lw=1,alpha=0.4)
if len(lo)>5: axes[0,0].plot(st,uniform_filter1d(lo,5),'royalblue',lw=2.5,label='Smoothed')
axes[0,0].set(xlabel='Step',ylabel='Loss',title='GRPO Training Loss'); axes[0,0].legend(); axes[0,0].grid(True,alpha=0.3)

# 2. Reward curve with baseline
if rs:
    axes[0,1].plot(rs,rw,'forestgreen',lw=1,alpha=0.4)
    if len(rw)>5: axes[0,1].plot(rs,uniform_filter1d(rw,5),'forestgreen',lw=2.5,label='Smoothed')
    axes[0,1].axhline(baseline['mean'],color='red',ls='--',lw=1.5,label=f"Baseline ({baseline['mean']:+.2f})")
    axes[0,1].legend()
axes[0,1].set(xlabel='Step',ylabel='Reward',title='Environment Reward'); axes[0,1].grid(True,alpha=0.3)

# 3. Before vs After bars
labels=['Mean\nReward','Positive\nRate','Valid\nJSON']
bv=[baseline['mean'],baseline['pos_rate'],baseline['json_rate']]
av=[trained['mean'],trained['pos_rate'],trained['json_rate']]
x=np.arange(3); w=0.35
b1=axes[1,0].bar(x-w/2,bv,w,label='Before',color='#e74c3c',alpha=0.85)
b2=axes[1,0].bar(x+w/2,av,w,label='After',color='#2ecc71',alpha=0.85)
for bar in list(b1)+list(b2): axes[1,0].text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.01,f'{bar.get_height():.2f}',ha='center',fontsize=9)
axes[1,0].set_xticks(x); axes[1,0].set_xticklabels(labels)
axes[1,0].set(ylabel='Score',title='Before vs After GRPO'); axes[1,0].legend(); axes[1,0].grid(True,alpha=0.3,axis='y')

# 4. Action distribution
all_actions=set(list(baseline['actions'].keys())+list(trained['actions'].keys()))
act_labels=sorted(all_actions)
bv2=[baseline['actions'].get(a,0) for a in act_labels]
av2=[trained['actions'].get(a,0) for a in act_labels]
x2=np.arange(len(act_labels)); w2=0.35
axes[1,1].bar(x2-w2/2,bv2,w2,label='Before',color='#e74c3c',alpha=0.85)
axes[1,1].bar(x2+w2/2,av2,w2,label='After',color='#2ecc71',alpha=0.85)
axes[1,1].set_xticks(x2); axes[1,1].set_xticklabels(act_labels,rotation=30,ha='right',fontsize=8)
axes[1,1].set(ylabel='Count',title='Action Type Distribution'); axes[1,1].legend(); axes[1,1].grid(True,alpha=0.3,axis='y')

plt.suptitle(f'EduPath AI \u2014 GRPO Training Results ({MODEL_ID})',fontsize=14,fontweight='bold')
plt.tight_layout()
plt.savefig('grpo_training_results.png',dpi=150,bbox_inches='tight')
plt.show()
print('Saved: grpo_training_results.png \u2705')

In [ ]:
# Save final model + print summary
model.eval()
trainer.save_model('edupath_grpo_final')
tokenizer.save_pretrained('edupath_grpo_final')

# Show qualitative examples
print('Trained model sample outputs:\n')
inp=tokenizer(eval_prompts[0],return_tensors='pt',truncation=True,max_length=512).to(model.device)
for i in range(5):
    with torch.no_grad():
        out=model.generate(**inp,max_new_tokens=64,temperature=0.7,do_sample=True,pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    text=tokenizer.decode(out[0][inp['input_ids'].shape[1]:],skip_special_tokens=True).strip()
    r=edupath_reward([text])[0]
    print(f'  {i+1}. {text[:80]:80s} r={r:+.2f}')

print(f'\n{"="*55}')
print(f'  FINAL SUMMARY')
print(f'  Model:  {MODEL_ID}')
print(f'  Steps:  {config.max_steps} | Time: {mins:.1f}min')
print(f'  Before: reward={baseline["mean"]:+.3f} positive={baseline["pos_rate"]:.0%} json={baseline["json_rate"]:.0%}')
print(f'  After:  reward={trained["mean"]:+.3f} positive={trained["pos_rate"]:.0%} json={trained["json_rate"]:.0%}')
print(f'  Delta:  {trained["mean"]-baseline["mean"]:+.3f}')
print(f'  Parse failures during training: {_parse_fails}')
print(f'{"="*55}')
print('Model saved to edupath_grpo_final/ \u2705')

## Summary
1. **Environment as Verifier**: BKT student simulation scores LLM tutoring actions
2. **Diverse dataset**: 400 train + 100 held-out eval across 5 student profiles
3. **Baseline recorded**: Untrained model evaluated on held-out set
4. **GRPO Training**: 500 steps with checkpoints every 100
5. **Evidence**: Loss/reward curves, before/after bars, action distribution, qualitative examples

Loop: `student state \u2192 LLM action \u2192 env verifies \u2192 reward \u2192 GRPO update`